<a href="https://colab.research.google.com/github/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Run Cellpose-SAM

Adapted from Marius Pachitariu, Michael Rariden, Carsen Stringer and the notebook by Pradeep Rajasekhar, inspired by the [ZeroCostDL4Mic notebook series](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki)

[paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [code](https://github.com/MouseLand/cellpose)

### Make sure you are in the correct environment


In [ ]:
# Check GPU and instantiate model - will download weights.
import numpy as np
from cellpose import models, core, io, plot
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
import cv2 as cv 
import tifffile as tf
%matplotlib inline
from natsort import natsorted

io.logger_setup() # run this to get printing of progress

#Check if GPU access


In [ ]:
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

Input directory with your images:
- Note - For best accuracy and runtime performance, resize images so cells are less than 100 pixels across

In [ ]:
#Inputs
dirfolder = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/"
dirfolder = Path(dirfolder)

dirlist = [dir for dir in dirfolder.glob("*")]
display(dirlist)

from cellpose_functions import *

# *** change to your image extension ***
image_ext = ".tif"
#nchannels = 4



In [ ]:
img_files_test = load_sorted_directory_list(dirlist[-1])
maskdir = dirlist[-1] / "masks"
maskdir.mkdir(exist_ok=True)

display(img_files_test)
print(maskdir)

In [ ]:
for dir in dirlist:
    maskdir = dir / "masks"
    maskdir.mkdir(exist_ok=True)
    #nchannels = 4
    if dir.name != "20250328":#the folders that are already done
        continue
    else:
        # if dir.name == "20241112":
        #     nchannels = 3
        # else:
        #     nchannels = 4
        print("writing to: " + str(maskdir))
        files = load_sorted_directory_list(dir)
        
        save_mask_folder(files,maskdir,v2=True)



### set up the correct order for the image filenames - sort by location first, then channel
these functions moved to cellpose_functions
```python
def file_sort_key(filename):
  parts = filename.split("-")
  channel = parts[0][-1:] # get the last character of the first part
  location = parts[1]
  return (location,channel)

def plate_location(filename):
  parts = filename.split("-")
  pre_location = parts[1]
  location = pre_location.split(".")[0] # get the first part of the second part
  return location
  
#list all files
def sort_files(dir, image_ext):
  if not dir.exists():
    raise FileNotFoundError("directory does not exist")
  files = sorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name and "SUM" not in f.name],
                           key=lambda x: file_sort_key(x.name))```
 # sort by number in filename
  if(len(files)==0):
    raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
  else:
    return files
  
def print_files(files):
  for f in files:
    print(f.name)

def group_files_by_channel(files, nchannels=4):
  grouped = []
  for i in range(0,len(files),nchannels):
    grouped.append(files[i:i+nchannels])
  return grouped

def print_grouped_files(grouped):
  for i in range(len(grouped)):
    print(f"\n Group {i+1} of {len(grouped)}")
    for j in range(len(grouped[i])):
      item = grouped[i][j]
      print(" "+ item.name)
```

In [ ]:
files = img_files_test 
#files = sort_files(dir, image_ext)
grouped_files = group_files_by_channel(files)

#print_files(files)
print_grouped_files(grouped_files)
print(plate_location(files[-1].name))

print(get_nchannels(files))

## Run Cellpose-SAM on one image in folder

Here are some of the parameters you can change:

* ***flow_threshold*** is  the  maximum  allowed  error  of  the  flows  for  each  mask.   The  default  is 0.4.
    *  **Increase** this threshold if cellpose is not returning as many masks as you’d expect (or turn off completely with 0.0)
    *   **Decrease** this threshold if cellpose is returning too many ill-shaped masks.

* ***cellprob_threshold*** determines proability that a detected object is a cell.   The  default  is 0.0.
    *   **Decrease** this threshold if cellpose is not returning as many masks as you’d expect or if masks are too small
    *   **Increase** this threshold if cellpose is returning too many masks esp from dull/dim areas.

* ***tile_norm_blocksize*** determines the size of blocks used for normalizing the image. The default is 0, which means the entire image is normalized together.
  You may want to change this to 100-200 pixels if you have very inhomogeneous brightness across your image.



In [ ]:
image_set_index = 1
in_channels = load_image_set(grouped_files[image_set_index])
set_name = get_image_set_name(grouped_files[image_set_index])
print("Set name: ", set_name)
display(in_channels)

img1 = img_preprocessing(in_channels)
img2 = img_rescaled(img1, factor=0.25)
tf.imshow(img1)
tf.imshow(img2)

In [ ]:
## For stitched images
image_ext = ".ome.tif"

stitched_path = Path("/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05/test_stitching/r02c02/Block_A/ashalr_img")
stitched_file = Path.joinpath(stitched_path,"r02c02_blockA"+image_ext)


stitched_img = io.imread(stitched_file)
#set_name = get_image_set_name(stitched_files)
#print("Set name: ", set_name)
#display(in_channels)

img1 = img_preprocessing(stitched_img)
img2 = img_rescaled(img1, factor=0.25)
tf.imshow(img1)
tf.imshow(img2)

In [ ]:
img = img2
from skimage import feature, filters   
cell_masks = segment_cell(img,model)
nuc_masks = segment_nuclei(img,model)
cell_v2_masks = segment_cell_v2(img,model)

#smoothed_masks = filters.unsharp_mask(cell_masks, radius=1).astype(uint)
#smoothed_masks = (smoothed_masks > 0.5).astype(int)  # Binarize the smoothed masks

save_masks(set_name+"diam", cell_masks, outdir=maskdir, image_ext=image_ext)
print(maskdir)
#save_masks(set_name+"smooth", smoothed_masks, outdir=maskdir, image_ext=image_ext)
#save_masks(set_name, nuc_masks, outdir=maskdir, image_ext=image_ext, mask_type="nuclei")
#blobs = filters.median(img[:,:,1])
#tf.imshow(blobs)

### Channel Selection
If you have a fluroescent image with multiple stains, you should choose one channel with a cytoplasm/membrane stain, one channel with a nuclear stain, and set the third channel to None. Choosing multiple channels may produce segmentaiton of all the structures in the image. If you have retrained the model on your data with a thrid stain (described below), you can run segmentation with all channels.

In [ ]:

#img = io.imread(files[0])
def segment_cell_tweaking(img, show=True):
    from skimage import exposure,filters,morphology
    gfp = img[:,:,0] #combine the ch1 and ch2 images to help cellpose out a bit
    rfp = img[:,:,1]
    dapi = img[:,:,2] #save ch3 for later
    
    img_combo = gfp+rfp
    img_combo = img_01_normalization(img_combo) #normalize to match cellpose training data
    #adjust contrast
    img_combo = exposure.equalize_adapthist(img_combo, kernel_size=68, clip_limit=0.01)
    #smooth and subtract background
    dog = filters.difference_of_gaussians(img_combo, low_sigma=2.5)
    img_combo = img_combo - dog
    
    #sharpen image and improve outline
    img_combo = filters.unsharp_mask(img_combo, radius=2, amount=1)
    
    #stack the images
    img_selected_channels = np.stack([img_combo, dapi],axis=-1)
    
    """ fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12,5),sharex=True, sharey=True)
    ax1.imshow(img_selected_channels[:,:,0])
    ax2.imshow(img_selected_channels[:,:,1]) """
     
    flow_threshold = 0.6
    cellprob_threshold = -1
    tile_norm_blocksize = 0
    diameter = 40

    masks, flows, styles = model.eval(img_selected_channels, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    #plot if true
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks
    
def segment_nuclei_tweaking(orig_img, show=True):
    from skimage import morphology, filters
    img = orig_img[:,:,2] # get the DAPI channel
    
    # remove background
    dog = filters.difference_of_gaussians(img, low_sigma=2.5)
    seed = np.minimum(dog, img)  # ensure seed is not greater than the original image
    bg = morphology.reconstruction(seed, img, method='dilation')
    img = img - bg
    
    # remove speckle-shaped autofluor
    bg2 = morphology.white_tophat(img, morphology.disk(3))
    img = img - bg2
    img = morphology.closing(img, morphology.disk(2.5))
    img = filters.gaussian(img, sigma=1)
    #img = img_01_normalization(img)
    
    flow_threshold = 0.5
    cellprob_threshold = 0
    tile_norm_blocksize = 0
    diameter = None

    masks, flows, styles = model.eval(img, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks

cell_masks = segment_cell_tweaking(img)
nuc_masks = segment_nuclei_tweaking(img)
save_masks(set_name, cell_masks, outdir=maskdir, image_ext=image_ext)
save_masks(set_name, nuc_masks, outdir=maskdir, image_ext=image_ext, mask_type="nuclei") 


In [ ]:

#img = io.imread(files[0])
def segment_cell_tweaking(img, show=True):
    from skimage import exposure,filters,morphology
    ch1 = img[:,:,0]
    ch2 = img[:,:,1]
    ch3 = img[:,:,2]
    #combine the ch1 and ch2 images to help cellpose out a bit
    img_combo = ch1+ch2
    img_combo = img_01_normalization(img_combo) #normalize to match cellpose training data
    img_combo = exposure.equalize_adapthist(img_combo, kernel_size=150, clip_limit=0.01)
    dog = filters.difference_of_gaussians(img_combo, low_sigma=2.5)
    
    #img_combo = filters.rank.median(img_combo, footprint=morphology.disk(1.5))
    
    img_combo = img_combo - dog
    #img_combo = filters.median(img_combo, morphology.disk(1.5))
    img_combo = filters.unsharp_mask(img_combo, radius=5, amount=2)
    #img_combo = filters.unsharp_mask(img_combo, radius=10, amount=2)
    img_selected_channels = np.stack([img_combo, ch3],axis=-1)
    
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12,5),sharex=True, sharey=True)
    ax1.imshow(img_selected_channels[:,:,0])
    ax2.imshow(img_selected_channels[:,:,1])
     
    flow_threshold = 0.5
    cellprob_threshold = -0.5
    tile_norm_blocksize = 0
    diameter = None

    masks, flows, styles = model.eval(img_selected_channels, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    #plot if true
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks
    
def segment_nuclei_tweaking(orig_img, show=True):
    from skimage import morphology, filters
    img = orig_img[:,:,2] # get the DAPI channel
    
    # remove background
    dog = filters.difference_of_gaussians(img, low_sigma=2.5)
    seed = np.minimum(dog, img)  # ensure seed is not greater than the original image
    bg = morphology.reconstruction(seed, img, method='dilation')
    img = img - bg
    
    # remove speckle-shaped autofluor
    bg2 = morphology.white_tophat(img, morphology.disk(3))
    img = img - bg2
    img = morphology.closing(img, morphology.disk(2.5))
    img = filters.gaussian(img, sigma=1)
    #img = img_01_normalization(img)
    
    flow_threshold = 0.4
    cellprob_threshold = 0
    tile_norm_blocksize = 0
    diameter = None

    masks, flows, styles = model.eval(img, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks

cell_masks = segment_cell_tweaking(img)
nuc_masks = segment_nuclei_tweaking(img)
save_masks(set_name, cell_masks, outdir=maskdir, image_ext=image_ext)
save_masks(set_name, nuc_masks, outdir=maskdir, image_ext=image_ext, mask_type="nuclei") 


## Run Cellpose-SAM on folder of images

if you have many large images, you may want to run them as a loop over images
See `cellpose_functions` to run this
 moved to `save_mask_folder() `


### loop for all files in the group in the directory
```python
for i in trange(len(grouped_files)):
    file_group = grouped_files[i]
    img_set = load_image_set(file_group)
    img_set_name = get_image_set_name(file_group)
    print("Set name: ", set_name)
    
    stacked_img = img_preprocessing(img_set)
    rescaled_img = img_rescaled(stacked_img, factor=0.25)
    
    cell_masks = segment_cell(rescaled_img, show=False)
    nuc_masks = segment_nuclei(rescaled_img, show=False) 
    
    save_masks(img_set_name, cell_masks, image_ext=image_ext)
    save_masks(img_set_name, nuc_masks, image_ext=image_ext, mask_type="nuclei") 
```